# Trabalho de APM - Classificação - G13

### 1. Importar bibliotecas

In [1]:
import warnings
warnings.filterwarnings("ignore")
warnings.simplefilter(action='ignore', category=FutureWarning)

import numpy as np
import pandas as pd

# Pré-processamento
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline

# Seleção de modelo
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.model_selection import GridSearchCV, ParameterGrid 

# Modelos
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier


from sklearn.svm import SVC

# Metricas
from sklearn.metrics import  confusion_matrix


### 2. Macro definições

In [2]:
# Bases de dados
DF_VEICULOS = "Veiculos - Dados.csv"
DF_DIABETES = "Diabetes - Dados.csv"

# Novos dados
DADOS_NOVOS_VEICULOS = "Veiculos - Novos Casos.csv"
DADOS_NOVOS_DIABETES = "Diabetes - Novos Casos.csv"

# Seed Global
RAND_SEED = 202613 # Ano atual + número do Grupo
np.random.seed(RAND_SEED)

# Tamanho da base de teste: 30%
TEST_SIZE = 0.3

# Largura máxima das colunas a serem exibidas
pd.set_option('display.max_colwidth', None)

### 3. Funções locais

In [3]:
# Carrega DataFrame
def load_df(df_name, column_to_drop=""):
    """Função para carregar o csv e se informado, remover colunas"""
    try:
        df = pd.read_csv(df_name, sep=',')
        if column_to_drop:
            df = df.drop(column_to_drop, axis=1)
        return df
    
    except:
        print("Erro ao processar o arquivo", df_name)

In [ ]:
# Função para executar o experimento

from sklearn.metrics import accuracy_score

def executar_experimento(config, X, y, classes, test_size=TEST_SIZE):

    modelo = config["modelo"]
    parametros = config["parametros"]
    
    # ======================================================
    # HOLD-OUT
    # ======================================================
    if config["avaliacao"] == "hold_out":

        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=test_size,
            stratify=y,
            random_state=RAND_SEED
        )

        melhor_score = -1
        melhor_param = None
        melhor_pipeline = None
        
        for p in ParameterGrid(parametros):
            modelo.set_params(**p)
            pipeline = Pipeline([("scaler", MinMaxScaler()), ("modelo", modelo)])
            pipeline.fit(X_train, y_train)

            y_pred = pipeline.predict(X_test)
            score = accuracy_score(y_test, y_pred)

            if score > melhor_score:
                melhor_score = score
                melhor_param = p
                melhor_pipeline = pipeline
                melhor_y_pred = y_pred

        return {
            "melhor_parametro": melhor_param,
            "score": round(melhor_score, 2),
            "modelo": melhor_pipeline,
            "confusion_matrix": confusion_matrix(y_test, melhor_y_pred, labels=classes) # calcula a matriz de confusão
        }

    # ======================================================
    # CROSS-VALIDATION
    # ======================================================
    elif config["avaliacao"] == "cross_validation":
        #Ajuste do nome dos parametros devido a passagem pelo pipeline
        param_grid = {
                    f'modelo__{k}': v
                    for k, v in config["parametros"].items()
                }
        
        pipeline = Pipeline([("scaler", MinMaxScaler()), ("modelo", modelo)])
        grid = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            cv=config.get("cv", 5),
            scoring=config["metrica"],
            n_jobs=-1
        )
        
        grid.fit(X, y)
        y_pred = cross_val_predict(grid.best_estimator_, X, y, cv=config.get("cv", 5)) # Predições fora da amostra

        return {
            "melhor_parametro": grid.best_params_,
            "score": round(grid.best_score_, 2),
            "modelo": grid.best_estimator_,
            "confusion_matrix": confusion_matrix(y, y_pred, labels=classes) # calcula a matriz de confusão
        }

    else:
        raise ValueError(
            f"Tipo de avaliação inválido: {config['avaliacao']}"
        )

In [5]:
# Função que treina os modelos para uma base de dados (X e y) e retorna o resultado do experimento

def testa_modelos(modelos, X, y, classes):
    # Dicionário para armazenar os resultados
    resultados = {}

    # Percorre todos os modelos configurados
    for chave, config in modelos.items():

        print(f"Executando: {config['nome']}")

        try:
            resultado = executar_experimento(config, X, y, classes)

            # Guarda o resultado completo
            resultados[chave] = {
                "nome": config["nome"],
                "avaliacao": config["avaliacao"],
                "metrica": config["metrica"],
                "melhor_parametro": resultado["melhor_parametro"],
                "score": resultado["score"],
                "modelo": resultado["modelo"],
                "confusion_matrix": resultado["confusion_matrix"]
            }

            print(f"  Melhor score: {resultado['score']:.2f}")
            print(f"  Melhor parâmetro: {resultado['melhor_parametro']}")

        except Exception as e:
            print(f"  Erro ao executar {config['nome']}: {e}")

        print("-" * 80)
    return resultados

### 4. Definição de Modelos

In [6]:

modelos = {
    "KNN": {
        "nome": "KNN",
        "modelo": KNeighborsClassifier(),
        "avaliacao": "hold_out",
        "parametros": {
                    'n_neighbors': [1, 3, 5, 7, 9],
        },
        "metrica": "accuracy"
    },
    
    "RNA_HO": {
        "nome": "RNA - Hold Out",
        "modelo": MLPClassifier(
            max_iter=2000,
            random_state=RAND_SEED
        ),
        "avaliacao": "hold_out",
        "parametros": {
            'hidden_layer_sizes': [(10,), (20,), (30, 10)],
            'learning_rate_init': [0.001, 0.01, 0.1]
        },
        "metrica": "accuracy"
    },
    
    "RNA_CV": {
        "nome": "RNA - Cross Validation",
        "modelo": MLPClassifier(
            max_iter=2000,
            random_state=RAND_SEED
        ),
        "avaliacao": "cross_validation",
        "parametros": {
            "hidden_layer_sizes": [(10,), (20,), (30, 10)],
            "learning_rate_init": [0.001, 0.01, 0.1]
        },
        "metrica": "accuracy"
    },

    "SVM_HO": {
        "nome": "SVM - Hold Out",
        "modelo": SVC(random_state=RAND_SEED),
        "avaliacao": "hold_out",
        "parametros": {
            'C': [1, 10, 50, 100],
            'gamma': ['auto', 'scale']
        },
        "metrica": "accuracy"
    },
    
    "SVM_CV": {
        "nome": "SVM - Cross Validation",
        "modelo": SVC(random_state=RAND_SEED),
        "avaliacao": "cross_validation",
        "parametros": {
            'C': [1, 10, 50, 100],
            'gamma': ['auto', 'scale']
        },
        "metrica": "accuracy"
    },
    
    "RF_HO": {
        "nome": "Random Forest - Hold Out",
        "modelo": RandomForestClassifier(
            random_state=RAND_SEED
        ),
        "avaliacao": "hold_out",
        "parametros": {
            "n_estimators": [10, 50, 100, 200],
            "max_depth": [None, 1, 10, 20]
        },
        "metrica": "accuracy"
    },
    
    "RF_CV": {
        "nome": "Random Forest - Cross Validation",
        "modelo": RandomForestClassifier(
            random_state=RAND_SEED
        ),
        "avaliacao": "cross_validation",
        "cv": 5,
        "parametros": {
            "n_estimators": [10, 50, 100, 200],
            "max_depth": [None, 1, 10, 20]
        },
        "metrica": "accuracy"
    }
}

In [7]:
# # Verificação
# for chave, config in modelos.items():
#     print(f"{chave}:")
#     print(f"  Nome       : {config['nome']}")
#     print(f"  Avaliação  : {config['avaliacao']}")
#     print(f"  Métrica    : {config['metrica']}")
#     print(f"  Parâmetros : {list(config['parametros'].keys())}")
#     print()

### 5. Carrega database de Veiculos e pré-processa

In [8]:
df = load_df(DF_VEICULOS, 'a')

y = df['tipo']
X = df.drop('tipo', axis = 1)

columns = list(X.columns)
classes = y.unique().tolist()

# # Restaura os nomes das colunas
X = pd.DataFrame(X, columns=columns)

### 6. Treina em diferentes modelos e armazena os resultados

In [9]:
resultados_veiculos = testa_modelos(modelos, X, y, classes)

Executando: KNN
  Melhor score: 0.70
  Melhor parâmetro: {'n_neighbors': 5}
--------------------------------------------------------------------------------
Executando: RNA - Hold Out
  Melhor score: 0.81
  Melhor parâmetro: {'hidden_layer_sizes': (30, 10), 'learning_rate_init': 0.001}
--------------------------------------------------------------------------------
Executando: RNA - Cross Validation
  Melhor score: 0.83
  Melhor parâmetro: {'modelo__hidden_layer_sizes': (30, 10), 'modelo__learning_rate_init': 0.001}
--------------------------------------------------------------------------------
Executando: SVM - Hold Out
  Melhor score: 0.82
  Melhor parâmetro: {'C': 10, 'gamma': 'scale'}
--------------------------------------------------------------------------------
Executando: SVM - Cross Validation
  Melhor score: 0.85
  Melhor parâmetro: {'modelo__C': 50, 'modelo__gamma': 'scale'}
--------------------------------------------------------------------------------
Executando: Random 

In [10]:
df_resultados_veiculos = pd.DataFrame.from_dict(resultados_veiculos, orient="index")

# Ordena pelo melhor score
df_resultados_veiculos = df_resultados_veiculos.sort_values("score", ascending=False)

print(df_resultados_veiculos[[
    "nome",
    "melhor_parametro",
    "score"
]].to_string(index=False))

                            nome                                                              melhor_parametro  score
          SVM - Cross Validation                                   {'modelo__C': 50, 'modelo__gamma': 'scale'}   0.85
          RNA - Cross Validation {'modelo__hidden_layer_sizes': (30, 10), 'modelo__learning_rate_init': 0.001}   0.83
                  SVM - Hold Out                                                   {'C': 10, 'gamma': 'scale'}   0.82
                  RNA - Hold Out                 {'hidden_layer_sizes': (30, 10), 'learning_rate_init': 0.001}   0.81
Random Forest - Cross Validation                        {'modelo__max_depth': 10, 'modelo__n_estimators': 100}   0.76
        Random Forest - Hold Out                                      {'max_depth': None, 'n_estimators': 200}   0.75
                             KNN                                                            {'n_neighbors': 5}   0.70


In [11]:
# Exibe a matriz de confusão de cada modelo
for idx, row in df_resultados_veiculos.iterrows():
    print(' ')
    print(idx)
    cm = row["confusion_matrix"]
    temp_df = pd.DataFrame(
        cm,
        index=classes,
        columns=classes)
    display(temp_df)

 
SVM_CV


,van,saab,bus,opel
van,198,1,0,0
saab,21,125,14,57
bus,6,0,212,0
opel,16,91,7,98


 
RNA_CV


,van,saab,bus,opel
van,194,1,3,1
saab,2,143,4,68
bus,5,1,212,0
opel,4,68,0,140


 
SVM_HO


...

,van,saab,bus,opel
van,197,1,1,0
saab,15,122,8,72
bus,3,0,214,1
opel,10,96,3,103


 
RF_HO


,van,saab,bus,opel
van,57,0,3,0
saab,9,37,2,17
bus,2,1,62,0
opel,1,27,1,35


 
KNN


,van,saab,bus,opel
van,52,0,6,2
saab,8,38,5,14
bus,0,1,61,3
opel,2,26,8,28


### 7. Faz predição para dados novos de Veículos usando o melhor modelo

In [12]:
# obtem o melhor modelo
best_model_name = df_resultados_veiculos["nome"].iloc[0]
best_model = df_resultados_veiculos["modelo"].iloc[0]

print(best_model_name)

# carrega novos dados
X_novo = load_df(DADOS_NOVOS_VEICULOS)

# faz a predição
y_novo = best_model.predict(X_novo)
print(y_novo.tolist())


SVM - Cross Validation
['van', 'van', 'saab']


### 8. Carrega database de Diabetes e pré-processa

In [13]:
df = load_df(DF_DIABETES, 'num')

y = df['diabetes']
X = df.drop('diabetes', axis = 1)

classes = y.unique().tolist()

### 9. Treina em diferentes modelos e armazena os resultados

In [14]:
resultados_diabetes = testa_modelos(modelos, X, y, classes)

Executando: KNN
  Melhor score: 0.73
  Melhor parâmetro: {'n_neighbors': 5}
--------------------------------------------------------------------------------
Executando: RNA - Hold Out
  Melhor score: 0.76
  Melhor parâmetro: {'hidden_layer_sizes': (10,), 'learning_rate_init': 0.1}
--------------------------------------------------------------------------------
Executando: RNA - Cross Validation
  Melhor score: 0.77
  Melhor parâmetro: {'modelo__hidden_layer_sizes': (20,), 'modelo__learning_rate_init': 0.01}
--------------------------------------------------------------------------------
Executando: SVM - Hold Out
  Melhor score: 0.73
  Melhor parâmetro: {'C': 10, 'gamma': 'auto'}
--------------------------------------------------------------------------------
Executando: SVM - Cross Validation
  Melhor score: 0.78
  Melhor parâmetro: {'modelo__C': 1, 'modelo__gamma': 'scale'}
--------------------------------------------------------------------------------
Executando: Random Forest - Ho

In [15]:
df_resultados_diabetes = pd.DataFrame.from_dict(resultados_diabetes, orient="index")

# Ordena pelo melhor score
df_resultados_diabetes = df_resultados_diabetes.sort_values("score", ascending=False)

print(df_resultados_diabetes[[
    "nome",
    "melhor_parametro",
    "score"
]].to_string(index=False))

                            nome                                                          melhor_parametro  score
          SVM - Cross Validation                                {'modelo__C': 1, 'modelo__gamma': 'scale'}   0.78
Random Forest - Cross Validation                     {'modelo__max_depth': 10, 'modelo__n_estimators': 50}   0.78
          RNA - Cross Validation {'modelo__hidden_layer_sizes': (20,), 'modelo__learning_rate_init': 0.01}   0.77
                  RNA - Hold Out                  {'hidden_layer_sizes': (10,), 'learning_rate_init': 0.1}   0.76
        Random Forest - Hold Out                                   {'max_depth': None, 'n_estimators': 10}   0.74
                             KNN                                                        {'n_neighbors': 5}   0.73
                  SVM - Hold Out                                                {'C': 10, 'gamma': 'auto'}   0.73


In [16]:
# Exibe a matriz de confusão de cada modelo
for idx, row in df_resultados_diabetes.iterrows():
    print(' ')
    print(idx)
    cm = row["confusion_matrix"]
    temp_df = pd.DataFrame(
        cm,
        index=classes,
        columns=classes)
    display(temp_df)

 
SVM_CV


,pos,neg
pos,152,116
neg,55,445


 
RF_CV


,pos,neg
pos,152,116
neg,64,436


 
RNA_CV


...

,pos,neg
pos,48,33
neg,27,123


 
KNN


,pos,neg
pos,48,33
neg,30,120


 
SVM_HO


,pos,neg
pos,44,37
neg,25,125


### 10. Faz predição para dados novos de Diabetes usando o melhor modelo

In [17]:
# obtem o melhor modelo
best_model_name = df_resultados_diabetes["nome"].iloc[0]
best_model = df_resultados_diabetes["modelo"].iloc[0]

print(best_model_name)

# carrega novos dados
X_novo = load_df(DADOS_NOVOS_DIABETES)

# faz a predição
y_novo = best_model.predict(X_novo)
print(y_novo.tolist())

SVM - Cross Validation
['neg', 'neg', 'pos']
